In [32]:
import sys
from pathlib import Path

# Get the absolute path of the parent directory of your current working directory
ROOT = Path.cwd().resolve().parent
# Add this parent directory to the top of Python's module search path
sys.path.insert(0, str(ROOT))


In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_ACR.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

wb = load_workbook(arquivo, read_only=True, data_only=True)
abas = [
    ws.title
    for ws in wb.worksheets
    if ws.sheet_state == "visible"
]

if "ACR" in arquivo.stem:
    bancos = ["SICOOB", "CAIXA"]
    extratos = [aba for aba in abas for banco in bancos if banco in aba]
    for extrato in extratos:
        print(f"Processando a planilha '{extrato}' do arquivo '{arquivo.name}'...")
        df = pd.read_excel(arquivo, sheet_name=extrato)

        if "SICOOB" in extrato:
            df.columns = df.iloc[1] # Definindo cabeçalho das colunas
            df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS / CONT']} {x['OBS / INT']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
        elif "CAIXA" in extrato:
            df.columns = df.iloc[4] # Definindo cabeçalho das colunas
            df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']}", axis=1)

            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.replace("None", "", regex=False).str.strip().str.upper()
            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].replace("", pd.NA)
            df = df.dropna(subset=["DESCRIÇÃO"])

        if df.empty:
            print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
            continue

        df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
        df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
        df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
        df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
        df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
        df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
        df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
        df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
        df["VALOR"] = df["VALOR"].abs()
        df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

        if not df.empty:
            arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
            shutil.copy2(origem_lanc, arquivo_lanc)
            planilha_lancamento(df, arquivo_lanc)

display(df)

['RESUMO', 'FATXREC', 'SICOOB - 18.713-5', 'CAIXA', 'INFORMAÇÕES DIVERSAS', 'ADIANT FORNECEDOR', 'CONTROLE PATRIMONIAL', 'CLIENTES - INADIMPLENTES']
Processando a planilha 'SICOOB - 18.713-5' do arquivo '0626_FECFIN_ACR.xlsx'...


C:\Users\manja\AppData\Local\Temp\ipykernel_14068\1648667339.py:52: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")


Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\[LANC] 0626_FECFIN_ACR_SICOOB - 18.713-5.xlsm
Processando a planilha 'CAIXA' do arquivo '0626_FECFIN_ACR.xlsx'...


4,DATA,DESCRIÇÃO,VALOR,TIPO


In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_ACR.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

wb = load_workbook(arquivo, read_only=True, data_only=True)
abas = [
    ws.title
    for ws in wb.worksheets
    if ws.sheet_state == "visible"
]

print(abas)

if "CEMAF 60" in arquivo.stem:
    bancos = ["SICOOB", "CAIXA"]
    extratos = [aba for aba in abas for banco in bancos if banco in aba]
    for extrato in extratos:
            print(f"Processando a planilha '{extrato}' do arquivo '{arquivo.name}'...")
            df = pd.read_excel(arquivo, sheet_name=extrato)

            if "SICOOB" in extrato:
                df.columns = df.iloc[1] # Definindo cabeçalho das colunas
                df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['OBS / INT']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
            elif "CAIXA" in extrato:
                df.columns = df.iloc[4] # Definindo cabeçalho das colunas
                df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']}", axis=1)

                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.replace("None", "", regex=False).str.strip().str.upper()
                df["DESCRIÇÃO"] = df["DESCRIÇÃO"].replace("", pd.NA)
                df = df.dropna(subset=["DESCRIÇÃO"])

            if df.empty:
                print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
                continue

            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
            df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
            df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
            df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
            df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
            df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
            df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
            df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
            df["VALOR"] = df["VALOR"].abs()
            df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

            if not df.empty:
                arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
                shutil.copy2(origem_lanc, arquivo_lanc)
                planilha_lancamento(df, arquivo_lanc)

display(df)

['RESUMO', 'NFS FORNEC', 'FATXREC', 'SICOOB - 15619-1', 'CAIXA', 'ADIANT A FORNECEDOR', 'INFORMAÇÕES DIVERSAS ']
Processando a planilha 'SICOOB - 15619-1' do arquivo '0626_FECFIN_CEMAF 60.xlsx'...
Arquivo preenchido com sucesso: G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\[LANC] 0626_FECFIN_CEMAF 60_SICOOB - 15619-1.xlsm
Processando a planilha 'CAIXA' do arquivo '0626_FECFIN_CEMAF 60.xlsx'...


4,DATA,DESCRIÇÃO,VALOR,TIPO


In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_ADP PART.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

wb = load_workbook(arquivo, read_only=True, data_only=True)
abas = [
    ws.title
    for ws in wb.worksheets
    if ws.sheet_state == "visible"
]

if "ALOHA" in arquivo.stem:
    bancos = ["INTER", "CAIXA"]
    extratos = [aba for aba in abas for banco in bancos if banco in aba]

    for extrato in extratos:
        print(f"Processando a planilha '{extrato}' do arquivo '{arquivo.name}'...")
        df = pd.read_excel(arquivo, sheet_name=extrato)

        if "INTER" in extrato:
            df.columns = df.iloc[1] # Definindo cabeçalho das colunas
            df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['OBS INTERNA']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
        elif "CAIXA" in extrato:
            df.columns = df.iloc[4] # Definindo cabeçalho das colunas
            df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
            df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']} {x['OBS INT']}", axis=1)

        if df.empty:
            print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
            continue

        df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
        df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
        df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
        df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
        df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
        df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
        df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
        df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
        df["VALOR"] = df["VALOR"].abs()
        df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

        if not df.empty:
            arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
            shutil.copy2(origem_lanc, arquivo_lanc)
            planilha_lancamento(df, arquivo_lanc)

    display(df)

In [ ]:
import shutil
import re
import pandas as pd
from pathlib import Path
from src.utils.helpers import planilha_lancamento

arquivo = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT\0626_FECFIN_CE PART.xlsx")
origem_lanc = Path(r"C:\Users\manja\OneDrive\Documentos\AutoExtrato\data\Lancamentos_Contabeis.xlsm")
destino_lanc = Path(r"G:\.shortcut-targets-by-id\1w3AY5xp-hHyQh263jfvhGoUdzH-0Ls0F\AUTOEXT\EXT")

with pd.ExcelFile(arquivo) as xls:
    abas = xls.sheet_names

    if "CE PART" in arquivo.stem:
        bancos = ["CORA", "CAIXA"]
        extratos = [aba for aba in abas for banco in bancos if banco in aba]

        for extrato in extratos:
            df = pd.read_excel(arquivo, sheet_name=extrato)

            if "CORA" in extrato:
                df.columns = df.iloc[1] # Definindo cabeçalho das colunas
                df = df[2:].reset_index(drop=True) # Removendo as primeiras 2 linhas e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['OBS']} {x['OBS INTERNA']} {x['TIPO']} {x['DOC']} {x['HISTÓRICO']}", axis=1)
            elif "CAIXA" in extrato:
                df.columns = df.iloc[4] # Definindo cabeçalho das colunas
                df = df[5:].reset_index(drop=True) # Removendo a primeira linha e resetando o índice
                df["DESCRIÇÃO"] = df.apply(lambda x: f"{x['HISTÓRICO']} {x['Nº DOC']} {x['TIPO']} {x['OBS']} {x['OBS INT']}", axis=1)
            
            if df.empty:
                print(f"A planilha '{extrato}' está vazia. Pulando para a próxima.")
                continue

            df["DESCRIÇÃO"] = df["DESCRIÇÃO"].str.replace("nan", "", regex=False).str.strip().str.upper()
            df = df[["DATA", "DESCRIÇÃO", "ENTRADA", "SAÍDA"]]
            df["ENTRADA"] = pd.to_numeric(df["ENTRADA"], errors="coerce").fillna(0)
            df["SAÍDA"] = pd.to_numeric(df["SAÍDA"], errors="coerce").fillna(0)
            df["VALOR"] = df["ENTRADA"].where(df["ENTRADA"] != 0, df["SAÍDA"] * -1)
            df = df.loc[~df["DESCRIÇÃO"].astype(str).str.upper().str.contains("SALDO", na=False)]
            df["TIPO"] = df["VALOR"].apply(lambda valor: "C" if valor > 0 else "D")
            df = df[["DATA", "DESCRIÇÃO", "VALOR", "TIPO"]]
            df["VALOR"] = df["VALOR"].abs()
            df["DATA"] = df["DATA"].apply(lambda x: pd.to_datetime(x, errors="coerce").strftime("%d/%m/%Y") if pd.notnull(x) else "")

            if not df.empty:
                arquivo_lanc = destino_lanc / f"[LANC] {arquivo.stem}_{extrato}.xlsm"
                shutil.copy2(origem_lanc, arquivo_lanc)
                planilha_lancamento(df, arquivo_lanc)